# Bank Marketing Campaign — Classification Model Comparison

**Assignment 17.1 | Practical Application: Classification**

## Business problem
A Portuguese bank conducts telephone marketing campaigns to encourage customers to subscribe to a term deposit. This binary classification project compares K-Nearest Neighbors (KNN), Logistic Regression, Decision Tree, and Support Vector Machine (SVM) models to help prioritize outreach.

The target is `y` (`yes` means subscription). Because positive outcomes are less common, accuracy alone can be misleading. We evaluate precision, recall, F1-score, ROC-AUC, and confusion matrices.

**Modeling decision:** `duration` is excluded because it is only known after the call, so it would not be available for pre-call targeting. See the [UCI Bank Marketing dataset](https://archive.ics.uci.edu/dataset/222/bank+marketing).

## 1. Imports and reproducibility

In [ ]:
import io
import zipfile
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Load the UCI dataset

The notebook first checks for `bank-additional-full.csv`. If absent, it downloads the official UCI ZIP archive and reads the semicolon-delimited CSV.

In [ ]:
DATA_FILE = Path("bank-additional-full.csv")
UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

if DATA_FILE.exists():
    df = pd.read_csv(DATA_FILE, sep=";")
else:
    try:
        with urllib.request.urlopen(UCI_ZIP_URL, timeout=60) as response:
            archive_bytes = response.read()
        with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
            matches = [name for name in archive.namelist()
                       if name.endswith("bank-additional-full.csv")]
            if not matches:
                raise FileNotFoundError("bank-additional-full.csv not found in UCI ZIP.")
            with archive.open(matches[0]) as file:
                df = pd.read_csv(file, sep=";")
    except Exception as exc:
        raise RuntimeError(
            "Dataset download failed. Download the UCI Bank Marketing ZIP, extract "
            "bank-additional-full.csv into the notebook directory, and rerun."
        ) from exc

print("Dataset shape:", df.shape)
display(df.head())

## 3. Data understanding and descriptive statistics

Inspect the schema, duplicate rows, target balance, and descriptive statistics. Categories named `unknown` are retained as categories rather than assumed to be missing.

In [ ]:
df.info()
display(df.describe(include="all").T)
print("Duplicate rows:", df.duplicated().sum())

target_counts = df["y"].value_counts(dropna=False)
target_pct = df["y"].value_counts(normalize=True, dropna=False).mul(100).round(2)
display(pd.DataFrame({"count": target_counts, "percent": target_pct}))

ax = target_counts.plot(kind="bar", title="Target distribution: term-deposit subscription")
ax.set_xlabel("Subscription (y)")
ax.set_ylabel("Number of customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Initial interpretation

After running the previous cell, describe the number of records, target-class balance, feature types, and duplicate count. Do not claim statistical significance from descriptive statistics alone.

## 4. Explore relationships with the target

These are descriptive associations, not proof of causality.

In [ ]:
for col in ["job", "marital", "education", "contact", "month", "poutcome"]:
    if col in df.columns:
        rates = (df.groupby(col, observed=False)["y"]
                   .apply(lambda s: (s == "yes").mean())
                   .sort_values(ascending=False))
        display((100 * rates).round(2).rename("subscription_rate_percent").to_frame())

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
display(df.groupby("y")[numeric_cols].median().T.rename_axis("feature"))

## 5. Prepare features and target

The target is `y`; `duration` is excluded because it is only known after the call. Numeric features are median-imputed and standardized. Categorical features are most-frequent-imputed and one-hot encoded. All transformations are inside a pipeline to avoid fitting preprocessing on the test set.

In [ ]:
if "y" not in df.columns:
    raise ValueError("Expected target column 'y' was not found.")

y = (df["y"].astype(str).str.lower() == "yes").astype(int)
X = df.drop(columns=["y", "duration"], errors="ignore").copy()

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Positive rate (train/test):", round(y_train.mean(), 4), round(y_test.mean(), 4))

## 6. Train the four classifiers

The same stratified split and preprocessing approach is used for all classifiers. KNN and RBF SVM can be slow on the full dataset; allow enough runtime.

In [ ]:
models = {
    "KNN": KNeighborsClassifier(n_neighbors=7, weights="distance", n_jobs=-1),
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8, min_samples_leaf=20, class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "SVM (RBF)": SVC(
        kernel="rbf", C=1.0, gamma="scale", class_weight="balanced",
        probability=False, cache_size=1000, random_state=RANDOM_STATE
    )
}

fitted_models, predictions, metric_rows = {}, {}, []

for name, classifier in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("classifier", classifier)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    score = (pipe.decision_function(X_test) if hasattr(pipe, "decision_function")
             else pipe.predict_proba(X_test)[:, 1])
    fitted_models[name] = pipe
    predictions[name] = pred
    metric_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision (yes)": precision_score(y_test, pred, zero_division=0),
        "Recall (yes)": recall_score(y_test, pred, zero_division=0),
        "F1 (yes)": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, score)
    })
    print("Finished:", name)

results = pd.DataFrame(metric_rows).set_index("Model").sort_values("F1 (yes)", ascending=False)
display(results.round(3))

## 7. Compare model performance

- **Precision (yes):** Among customers predicted to subscribe, the share who subscribed.
- **Recall (yes):** Among actual subscribers, the share identified by the model.
- **F1:** Harmonic mean of positive-class precision and recall.
- **ROC-AUC:** Ranking quality across classification thresholds.
- **Accuracy:** Overall correctness; it may obscure poor performance on the minority class.

The preferred model depends on campaign economics: high recall can help avoid missing potential subscribers, while higher precision can reduce calls to customers unlikely to subscribe.

In [ ]:
display(results.round(3).style.format("{:.3f}"))
results[["Precision (yes)", "Recall (yes)", "F1 (yes)", "ROC-AUC"]].plot(
    kind="bar", figsize=(11, 5), ylim=(0, 1), title="Classifier comparison"
)
plt.ylabel("Score")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 8. Confusion matrices and classification reports

In [ ]:
for name, pred in predictions.items():
    print("\\n" + "=" * 60)
    print(name)
    print(classification_report(
        y_test, pred, labels=[0, 1],
        target_names=["no subscription", "subscription"],
        digits=3, zero_division=0
    ))
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=["No", "Yes"], values_format="d"
    )
    plt.title(f"Confusion matrix — {name}")
    plt.tight_layout()
    plt.show()

## 9. Cross-validation and inferential interpretation

Test-set scores estimate performance on this specific holdout. A score difference on one split does not, by itself, prove statistical superiority. Stratified cross-validation on the training data provides a more stable comparison; hyperparameter tuning should remain within training data.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

for name, classifier in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("classifier", classifier)])
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring={"f1": "f1", "roc_auc": "roc_auc"},
        n_jobs=1, return_train_score=False
    )
    cv_rows.append({
        "Model": name,
        "CV F1 mean": scores["test_f1"].mean(),
        "CV F1 std": scores["test_f1"].std(),
        "CV ROC-AUC mean": scores["test_roc_auc"].mean(),
        "CV ROC-AUC std": scores["test_roc_auc"].std()
    })

cv_results = (pd.DataFrame(cv_rows).set_index("Model")
              .sort_values("CV F1 mean", ascending=False))
display(cv_results.round(3))

## 10. Business findings, recommendations, and next steps

**Complete this section after running the notebook. Use actual output values; do not invent model scores.**

### Findings to report
1. **Class balance:** Report the share of subscriptions and non-subscriptions. Explain why accuracy alone can mislead.
2. **Best model:** Select a model using the metric aligned to business costs, and support the choice with both holdout and cross-validation results.
3. **Operational errors:** Interpret false positives (wasted calls) and false negatives (missed possible subscriptions) using confusion matrices.
4. **Predictive signals:** Describe associations observed in the exploratory analysis without claiming they are causal.

### Recommendations
- Pilot a ranked outreach list rather than automatically excluding customers based on model predictions.
- Set the decision threshold using call capacity and the relative costs of false positives and false negatives.
- Measure incremental conversion and cost per subscription against a randomized control group.
- Monitor performance over time and check for drift.

### Limitations and next steps
- `duration` is excluded because it is not known before the call.
- The historical data comes from one Portuguese bank; validate before applying it to another population.
- Review privacy and fairness implications before operational use.
- Consider probability calibration, threshold tuning, cost-sensitive evaluation, and ongoing monitoring.
- Keep the test set untouched until model selection is complete.

## References

- Moro, S., Cortez, P., & Rita, P. (2014). *A Data-Driven Approach to Predict the Success of Bank Telemarketing*. Decision Support Systems.
- UCI Machine Learning Repository: [Bank Marketing dataset](https://archive.ics.uci.edu/dataset/222/bank+marketing).
- scikit-learn: [Pipelines and preprocessing](https://scikit-learn.org/stable/modules/compose.html), [classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html).

## Submission checklist
- [ ] Run every cell from top to bottom.
- [ ] Review full-dataset runtime, especially KNN and RBF SVM.
- [ ] Replace the interpretation prompts with findings from the actual results.
- [ ] Confirm there are no errors and charts render.
- [ ] Submit the public repository URL in the assignment portal.